In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & DATABASE CONNECTION
# =============================================================================
import sys
import os
import importlib
import pandas as pd
import numpy as np
import time
import datetime
from sqlalchemy import create_engine, text

# Add the parent directory to sys.path to allow importing from main directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Import and reload database initialization module
import init_database
importlib.reload(init_database)

# Establish connection
engine = init_database.init_db()

print("✅ Environment ready and database connection established!")

In [ ]:
def load_silver(engine):
    try:
        batch_start_time = time.time()
        print('================================================')
        print('Loading Silver Layer')
        print('================================================')

        # --- 1. CRM TABLES ---
        print('------------------------------------------------')
        print('Loading CRM Tables')
        print('------------------------------------------------')

        # A) silver.crm_cust_info
        start_time = time.time()
        print('>> Processing: silver.crm_cust_info')
        df_cust = pd.read_sql("SELECT * FROM bronze.crm_cust_info", engine)
        
        # A) CRM Customers: Deduplication & Standardization
        df_cust['cst_create_date'] = pd.to_datetime(df_cust['cst_create_date'])
        df_cust = df_cust.loc[df_cust['cst_id'].notnull()].copy()
        # Keep only the latest record per customer (SCD Type 1 logic)
        df_cust = df_cust.sort_values(['cst_id', 'cst_create_date'], ascending=[True, False])
        df_cust = df_cust.loc[~df_cust.duplicated('cst_id', keep='first')].copy()
        
        # Text cleaning and mapping
        df_cust['cst_firstname'] = df_cust['cst_firstname'].str.strip()
        df_cust['cst_lastname'] = df_cust['cst_lastname'].str.strip()
        
        m_map = {'S': 'Single', 'M': 'Married'}
        df_cust['cst_marital_status'] = df_cust['cst_marital_status'].str.strip().str.upper().map(m_map).fillna('n/a')
        
        g_map = {'F': 'Female', 'M': 'Male'}
        df_cust['cst_gndr'] = df_cust['cst_gndr'].str.strip().str.upper().map(g_map).fillna('n/a')
        
        df_cust['dwh_create_date'] = datetime.datetime.now()
        df_cust.to_sql('crm_cust_info', engine, schema='silver', if_exists='replace', index=False)
    
        row_count = len(df_cust)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        # B) silver.crm_prd_info
        start_time = time.time()
        print('>> Processing: silver.crm_prd_info')
        df_prd_info = pd.read_sql("SELECT * FROM bronze.crm_prd_info", engine)

        # Extract Category and Product Keys
        df_prd_info['cat_id'] = df_prd_info['prd_key'].str[:5].str.replace('-', '_', regex=False)
        df_prd_info['prd_key'] = df_prd_info['prd_key'].str[6:]
        df_prd_info['prd_cost'] = df_prd_info['prd_cost'].fillna(0)

        # Map Product Lines
        line_map ={
        'R': 'Road',
        'M': 'Mountain',
        'S': 'Other Sales',
        'T': 'Touring'
        }
        df_prd_info['prd_line'] = df_prd_info['prd_line'].str.strip().str.upper().map(line_map).fillna('n/a')

        # Correct Date overlapping
        df_prd_info['prd_start_dt'] = pd.to_datetime(df_prd_info['prd_start_dt'], errors='coerce')
        df_prd_info['prd_end_dt'] = pd.to_datetime(df_prd_info['prd_end_dt'], errors='coerce')
        df_prd_info['prd_end_dt'] = df_prd_info.groupby('prd_key')['prd_start_dt'].shift(-1) - pd.Timedelta(days=1)

        df_prd_info['dwh_create_date'] = datetime.datetime.now()
        df_prd_info.to_sql('crm_prd_info', engine, schema='silver', if_exists='replace', index=False)

        row_count = len(df_prd_info)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        # C) silver.crm_sales_details
        start_time = time.time()
        print('>> Processing: silver.crm_sales_details')
        df_sales = pd.read_sql("SELECT * FROM bronze.crm_sales_details", engine)

        # Convert Dates
        for col in ['sls_order_dt', 'sls_ship_dt', 'sls_due_dt']:
            df_sales[col] =pd.to_datetime(df_sales[col].astype(str), format= '%Y%m%d', errors='coerce')

        # Recalculate Sales (Sales = Qty * Price)
        df_sales['sls_sales'] = np.where(
            (df_sales['sls_sales'].isna())|
            (df_sales['sls_sales'] <= 0)|
            (df_sales['sls_sales'] != df_sales['sls_quantity'] * df_sales['sls_price'].abs()),
            df_sales['sls_quantity'] * df_sales['sls_price'].abs(),
            df_sales['sls_sales']
        )

        # Derive Price if missing
        df_sales['sls_price'] = np.where(
            (df_sales['sls_price'].isna())|
            (df_sales['sls_price'] <= 0)|
            (df_sales['sls_price'] != df_sales['sls_sales'] / df_sales['sls_quantity'].abs().replace(0, np.nan)),
            df_sales['sls_sales'] / df_sales['sls_quantity'].abs(),
            df_sales['sls_price']
        )

        df_sales['dwh_create_date'] = datetime.datetime.now()
        df_sales.to_sql('crm_sales_details', engine, schema='silver', if_exists='replace', index=False)

        row_count = len(df_sales)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

            # --- 2. ERP TABLES ---
        print('------------------------------------------------')
        print('Loading ERP Tables')
        print('------------------------------------------------')

        # A) silver.erp_cust_az12
        start_time = time.time()
        print('>> Processing: silver.erp_cust_az12')
        df_erp_cust = pd.read_sql("SELECT * FROM bronze.erp_cust_az12", engine)

        # Clean CID by removing 'NAS' prefix and any dashes
        df_erp_cust['CID'] = df_erp_cust['CID'].str.replace('^NAS', '', regex=True)

        # Convert BDATE to datetime and set future dates to NaT
        df_erp_cust['BDATE'] = pd.to_datetime(df_erp_cust['BDATE'], errors='coerce')
        df_erp_cust.loc[df_erp_cust['BDATE'] > pd.Timestamp.today()] = pd.NaT

        # Standardize GEN values
        df_erp_cust['GEN']= df_erp_cust['GEN'].str.strip().str.upper()
        df_erp_cust['GEN']= np.where(df_erp_cust['GEN'].isin(['F','FEMALE']), 'Female',
                        np.where(df_erp_cust['GEN'].isin(['M', 'MALE']), 'Male', 'n/a'))
        
        df_erp_cust['dwh_create_date'] = datetime.datetime.now()

        df_erp_cust.to_sql('erp_cust_az12', engine, schema='silver', if_exists='replace', index=False)

        row_count = len(df_erp_cust)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        # B) silver.erp_loc_a101
        start_time = time.time()
        print('>> Processing: silver.erp_loc_a101')
        df_erp_loc = pd.read_sql("SELECT * FROM bronze.erp_loc_a101", engine)

        # Clean CID by removing dashes
        df_erp_loc['CID'] = df_erp_loc['CID'].str.replace('-', '', regex=False)

        # Standardize Country Codes to Full Names
        country_map = {
        'US': 'United States',
        'USA': 'United States',
        'UNITED STATES': 'United States',
        'UK': 'United Kingdom',
        'UNITED KINGDOM': 'United Kingdom',
        'DE': 'Germany',
        'GERMANY': 'Germany',
        'FR': 'France',
        'FRANCE': 'France',
        'CA': 'Canada',
        'CANADA': 'Canada',
        'AU': 'Australia',
        'AUSTRALIA': 'Australia',
        '': 'n/a'
        }
        df_erp_loc['CNTRY'] = df_erp_loc['CNTRY'].astype(str).str.strip().str.upper().map(country_map).fillna('n/a')

        df_erp_loc['dwh_create_date'] = datetime.datetime.now()

        df_erp_loc.to_sql('erp_loc_a101', engine, schema='silver', if_exists='replace', index=False)

        row_count = len(df_erp_loc)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        # C) silver.erp_px_cat_g1v2
        start_time = time.time()
        print('>> Processing: silver.erp_px_cat_g1v2')
        df_erp_px = pd.read_sql("SELECT * FROM bronze.erp_loc_a101", engine) 

        df_erp_px['dwh_create_date'] = datetime.datetime.now()

        df_erp_px.to_sql('erp_px_cat_g1v2', engine, schema='silver', if_exists='replace', index=False) 

        row_count = len(df_erp_px)
        end_time = time.time()
        duration = round(end_time - start_time, 2)

        print(f'>> Row Loaded: {row_count}')
        print(f'>> Load Duration: {duration} seconds')
        print('>> -------------')

        print('==========================================')
        print(f'Loading Silver Layer Completed in {round(time.time()-batch_start_time, 2)} sec')
        print('==========================================')

    except Exception as e:
        print('==========================================')
        print('ERROR OCCURED DURING LOADING SILVER LAYER')
        print(f'Error Message: {str(e)}')
        print('==========================================')

# Execute the process
if __name__ == "__main__":

    # If the engine is successfully created, proceed to load the bronze layer
    if engine:
        load_silver(engine)
    else:
        print("❌ Failed to connect to database. Aborting load.")
